<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-03-prompt-engineering/lesson-3.3-advanced-reasoning/notebooks/exercises-3.3.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 3.3: Advanced Reasoning — Practice Exercises

**Netsetos GenAI Course v2.0** | Module 3 | 8 exercises: CoT variants, self-consistency, reasoning models, structured CoT, decomposition, CoVe, cost routing, full benchmark.

---

In [ ]:
!pip install google-generativeai -q
import google.generativeai as genai
import time, re
# genai.configure(api_key='YOUR_KEY')
model = genai.GenerativeModel('gemini-2.0-flash')
def ask(prompt, temp=0.0):
    start = time.time()
    r = model.generate_content(prompt, generation_config={'temperature': temp})
    elapsed = time.time() - start
    return r.text, elapsed


---
## Exercise 1 (Easy): CoT vs Direct on GST Problems
Solve 5 GST word problems with and without chain-of-thought.

In [ ]:
problems = [
    ('Laptop ₹50,000 at 18% GST intra-state. Total?', '₹59,000'),
    ('Restaurant bill ₹2,000 at 5% GST. Total?', '₹2,100'),
    ('Gold ₹1,00,000 at 3% GST inter-state. IGST amount?', '₹3,000'),
    ('AC ₹35,000 at 28% GST. CGST + SGST split?', '₹4,900 each'),
    ('Medicine ₹500 at 12% GST. Total with GST?', '₹560'),
]

print('DIRECT vs CHAIN-OF-THOUGHT')
d_correct, c_correct = 0, 0
for q, exp in problems:
    d, d_t = ask(f'Answer briefly: {q}')
    c, c_t = ask(f'{q}\nThink step by step before answering.')
    print(f'\nQ: {q}')
    print(f'  Direct ({d_t:.1f}s): {d.strip()[:60]}')
    print(f'  CoT    ({c_t:.1f}s): {c.strip()[:80]}')
    print(f'  Expected: {exp}')


---
## Exercise 2 (Easy): Self-Consistency
Sample 5 chains, majority vote on a tricky math problem.

In [ ]:
problem = 'A shopkeeper buys 100 items at ₹80 each. Sells 60 at ₹100, 30 at ₹70, rest at ₹50. Profit or loss? How much?'

answers = []
for i in range(5):
    r, _ = ask(f'{problem}\nThink step by step.', temp=0.7)
    # Extract final number
    nums = re.findall(r'₹?([\d,]+)', r)
    answers.append(r.strip()[:100])
    print(f'Chain {i+1}: {r.strip()[:80]}')

print(f'\n5 chains sampled. Majority vote gives the most reliable answer.')
print('Self-Consistency: +17.9% accuracy on GSM8K (Wang et al. 2023)')


---
## Exercise 3 (Medium): Reasoning Model Comparison
Same problem on different models/configs.

In [ ]:
hard_q = 'In a chess tournament, 8 players each play every other player once. How many games total? If each game lasts 45 minutes with 15 minute breaks between games on 2 boards, what is the minimum hours to finish?'

# Standard model, no CoT
r1, t1 = ask(hard_q)
print(f'Standard (no CoT) [{t1:.1f}s]: {r1.strip()[:100]}')

# Standard model + CoT
r2, t2 = ask(f'{hard_q}\nThink step by step.')
print(f'Standard + CoT [{t2:.1f}s]: {r2.strip()[:100]}')

# Gemini thinking model (if available)
try:
    think_model = genai.GenerativeModel('gemini-2.5-flash-preview-04-17')
    r3 = think_model.generate_content(hard_q)
    print(f'Gemini 2.5 Flash thinking: {r3.text.strip()[:100]}')
except:
    print('Gemini 2.5 Flash not available in free tier — try in Vertex AI')

print(f'\nLatency: no CoT={t1:.1f}s | CoT={t2:.1f}s | reasoning model = auto')


---
## Exercise 4 (Medium): Structured CoT Pipeline
GST calculator with XML tags, parsed programmatically.

In [ ]:
prompt = '''Calculate GST for this invoice:
Items: 2 phones at ₹25,000 each, intra-state, 18% GST

Reason inside <thinking> tags. Give ONLY the final breakdown in <answer> tags as JSON.

<thinking>[your reasoning]</thinking>
<answer>{"base": ..., "cgst": ..., "sgst": ..., "total": ...}</answer>'''

result, _ = ask(prompt)
print('Full response:')
print(result[:300])

# Parse answer tag
match = re.search(r'<answer>(.*?)</answer>', result, re.DOTALL)
if match:
    answer = match.group(1).strip()
    print(f'\nParsed answer: {answer}')
else:
    print('\nNo <answer> tag found — model did not follow structure')

# Parse thinking tag
think_match = re.search(r'<thinking>(.*?)</thinking>', result, re.DOTALL)
if think_match:
    thinking = think_match.group(1).strip()
    print(f'\nReasoning trace ({len(thinking)} chars): {thinking[:100]}...')


---
## Exercise 5 (Medium): Prompt Decomposition
4-step chain for legal document analysis vs single prompt.

In [ ]:
doc = '''This agreement between ABC Corp (Seller) and XYZ Ltd (Buyer)
dated 15th March 2025 for supply of 1000 units of electronic components
at ₹500 per unit with 18% GST. Payment within 30 days of delivery.
Penalty of 2% per month for late payment. Governed by laws of Telangana.'''

# SINGLE PROMPT
single, t1 = ask(f'Analyze this contract completely: {doc}')
print(f'Single prompt [{t1:.1f}s]:')
print(single.strip()[:200])

# CHAIN: 4 focused calls
steps = [
    ('Extract parties, amounts, dates', 'Step 1 (extract)'),
    ('Identify key terms and obligations', 'Step 2 (terms)'),
    ('Flag potential risks or issues', 'Step 3 (risks)'),
    ('Generate 3-sentence executive summary', 'Step 4 (summary)'),
]

context = doc
total_t = 0
for instruction, label in steps:
    r, t = ask(f'{instruction}:\n{context}')
    total_t += t
    context += f'\n{label}: {r.strip()[:200]}'
    print(f'\n{label} [{t:.1f}s]: {r.strip()[:100]}')

print(f'\nSingle: {t1:.1f}s | Chain: {total_t:.1f}s')
print('Chain gives more structured, verifiable output despite more calls.')


---
## Exercise 6 (Hard): CoVe for Indian Law
Fact-check LLM-generated legal citations.

In [ ]:
# Step 1: Draft
draft, _ = ask('List 5 important sections of the Indian IT Act 2000 with descriptions.')
print('DRAFT:', draft.strip()[:200])

# Step 2: Generate fact-check questions
questions, _ = ask(f'Generate 5 yes/no fact-check questions about these claims:\n{draft}')
print(f'\nQUESTIONS: {questions.strip()[:200]}')

# Step 3: Answer INDEPENDENTLY (key: don't show original draft!)
verified, _ = ask(f'Answer these questions about the Indian IT Act 2000 independently:\n{questions}')
print(f'\nVERIFICATION: {verified.strip()[:200]}')

# Step 4: Revised response
final, _ = ask(f'Based on this verification, provide accurate IT Act sections:\n{verified}')
print(f'\nFINAL (verified): {final.strip()[:200]}')
print('\nCoVe: 4 calls, but hallucinated citations reduced by ~77%')


---
## Exercise 7 (Hard): Cost-Optimized Model Router
Classify query complexity, route to appropriate model.

In [ ]:
queries = [
    ('What is the capital of India?', 'simple'),
    ('Summarize this paragraph about AI', 'simple'),
    ('Calculate GST on ₹50000 at 18%', 'medium'),
    ('Compare 3 approaches to distributed caching', 'medium'),
    ('Prove that sqrt(2) is irrational', 'hard'),
    ('Design a microservices architecture for a banking app', 'hard'),
]

costs = {'simple': 0.10, 'medium': 0.30, 'hard': 10.0}  # $/M tokens
all_hard_cost = sum(costs['hard'] for _ in queries)
routed_cost = sum(costs[level] for _, level in queries)

print('QUERY ROUTING RESULTS:')
for q, level in queries:
    model_name = {'simple': 'Flash-Lite', 'medium': 'Gemini Flash', 'hard': 'o4-mini'}[level]
    print(f'  [{level:>6}] {model_name:<14} → "{q[:45]}"')

savings = (1 - routed_cost / all_hard_cost) * 100
print(f'\nAll-hard cost:  ${all_hard_cost:.2f}/M tokens')
print(f'Routed cost:    ${routed_cost:.2f}/M tokens')
print(f'Savings:        {savings:.0f}%')


---
## Exercise 8 (Challenge): Full Reasoning Benchmark
Compare 5 methods across 4 difficulty levels.

In [ ]:
print('''BENCHMARK DESIGN:\n')
print('METHODS: direct, zero-shot CoT, few-shot CoT, self-consistency (5x), reasoning model')
print('DIFFICULTY: easy (arithmetic), medium (GST), hard (logic), very hard (AIME-style)')
print()
print('EXPECTED RESULTS:')
print('  Easy:    Direct=95% | CoT=96% → CoT adds latency for nothing')
print('  Medium:  Direct=60% | CoT=82% → CoT essential (+22%)')
print('  Hard:    Direct=30% | CoT=55% | SC=68% → Self-consistency wins')
print('  V.Hard:  Direct=10% | CoT=25% | Reasoning=75% → Model > prompt')
print()
print('COST PER CORRECT ANSWER (key metric):')
print('  Best overall: Gemini 2.5 Flash with auto thinking')
print('  Best accuracy: o3 high effort')
print('  Best budget:   DeepSeek-R1 ($0.55/M) or QwQ-32B (free)')


---
**8 exercises complete.** 10 reasoning techniques mastered: CoT taxonomy, zero/few-shot CoT, self-consistency, reasoning models, when NOT to reason, structured CoT, decomposition, ToT, ReAct, Reflexion, verification chains, cost framework.